In [68]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # for older PyTorch
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"  # disable gpt.py kernels progress bars
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # for deterministic ops
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import sys
import torch
import pickle

from pathlib import Path
root = str((Path.cwd() / "../..").resolve())
sys.path.insert(0, root) if root not in sys.path else None

from nanorepro.dataloader import DataLoaderSFT
from nanorepro.tasks import TaskMixture, TaskSmolTalk, TaskMMLU, TaskArc, TaskGSM8K
from nanorepro.tasks import TaskSimpleSpelling, TaskSpellingBee, TaskCustomJSON
from nanorepro.loss_eval import evaluate_bpb
from nanorepro.checkpoint import load_model
from nanorepro.common import get_base_path

BASE_DIR = get_base_path()

In [2]:
class Args:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)
args = Args(
    run="scaling3_6e18_d16-counting",
    no_fa3=True,
    deterministic=True,
    eval_tokens=524288,   # 40*524288,
)

In [60]:
# Compute setup and helpers
device, ddp_master, ddp_rank, ddp_world_size = 'cuda', True, 0, 1
enable_fp8 = False
print0 = print if os.environ.get("RANK", "0") == "0" else lambda *args, **kwargs: None
synchronize = lambda: torch.cuda.synchronize() if device.startswith("cuda") else None
compute_dtype = torch.bfloat16
run_path = os.path.join(BASE_DIR, "runs_sft", args.run if args.run is not None else "default")

In [4]:
# Tokenizer
tok_base_path = os.path.join(BASE_DIR, "tokenizer")
tokenizer_path = os.path.join(tok_base_path, "tokenizer.pkl")
tokenizer = pickle.load(open(tokenizer_path, "rb"))
token_bytes_path = os.path.join(tok_base_path, "token_bytes.pt")
with open(token_bytes_path, "rb") as f:
    token_bytes = torch.load(f, map_location=device)
print0("Vocabulary size:", tokenizer.n_vocab)

Vocabulary size: 32768


In [5]:
# Reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)

# Precision
if device.startswith("cuda"):
    torch.set_float32_matmul_precision("high")  # uses tf32 instead of fp32 for matmuls

# Determinism
# Also need to disable torch.compile for reproducibility
if args.deterministic:
    assert args.no_fa3, "FA3 can't reliably be set to deterministic mode due to bug in upstream implementation"
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)

In [6]:
# Model Setup
run_name = args.run if args.run is not None else "default"
checkpoints_path = os.path.join(BASE_DIR, "runs_sft", run_name)
model, pretrain_metadata = load_model(
    checkpoints_path=checkpoints_path,
    compute_dtype=compute_dtype,
    enable_fa3=not args.no_fa3,
    fp8_training=enable_fp8,    # doesn't matter
    enable_metrics=False,
    device=device,
    step=None)                  # latest checkpoint
print0("Model configuration:")
for k, v in model.config.to_dict().items():
    print0(f"  {k:>16}: {v}")

Model configuration:
        block_size: 2048
        vocab_size: 32768
           n_layer: 16
            n_head: 8
            n_embd: 1024
    window_pattern: SSSL
        moe_enable: False
       moe_experts: 8
         moe_top_k: 2


In [7]:
# Compile
orig_model = model
if not args.deterministic:
    model = torch.compile(model, dynamic=False)

In [8]:
# Hyperparameter Transfer and Calculation
step = pretrain_metadata["step"]
pretrain_user_cfg = pretrain_metadata["user_config"]
max_seq_len = pretrain_user_cfg['max_seq_len']
micro_batch = pretrain_user_cfg['device_batch_size']
total_batch_size = pretrain_user_cfg["total_batch_size"]
flops_per_token = model.estimate_flops_per_token()

total_flops = step * total_batch_size * flops_per_token

In [9]:
# Eval Dataloader
assert args.eval_tokens % (micro_batch * max_seq_len * ddp_world_size) == 0
eval_steps = args.eval_tokens // (micro_batch * max_seq_len * ddp_world_size)
tasks_eval = TaskMixture([
    TaskSmolTalk(split="test"),                        # 24K tasks
    TaskMMLU(subset="all", split="test", stop=5200),   #  5.2K tasks - match training ratio before repetition (whole test set is 14K)
    TaskGSM8K(subset="main", split="test", stop=420),  #  0.42K tasks (whole test set is 1.32K)
])
eval_loader = DataLoaderSFT(
    tasks=tasks_eval,
    batch_size=micro_batch,
    block_size=max_seq_len,
    tokenizer=tokenizer,
    device=device,
)

In [10]:
# BPB Evaluation
bpb, total_nats, total_bytes = evaluate_bpb(model, token_bytes, eval_loader, eval_steps, device)
print0(f"BPB Eval {step} | BPB {bpb:.14f} | nats {total_nats:.1f} | bytes {total_bytes}")

BPB Eval 969 | BPB 0.31022950461913 | nats 397260.1 | bytes 1847423


In [11]:
from nanorepro.tokenizer import ConversationRenderer

In [ ]:
def evaluate_sft_categorical(task, model, tokenizer, renderer, device, micro_batch, max_seq_len, max_problems=None):
    """Evaluate a categorical SFT task, model selects one of small number of options (e.g. MMLU A,B,C,D)"""
    ddp_rank = torch.distributed.get_rank() if torch.distributed.is_initialized() else 0
    ddp_world_size = torch.distributed.get_world_size() if torch.distributed.is_initialized() else 1
    bos_token = renderer.bos_token
    assistant_start_token = renderer.assistant_start_token

    num_passed, num_total = 0, 0  # this rank
    num_world_total = len(task) if max_problems is None else min(max_problems, len(task))
    for start_i in range(ddp_rank * micro_batch, num_world_total, ddp_world_size * micro_batch):
        batch_size = min(micro_batch, num_world_total - start_i)

        # Built batch of examples
        batch_examples = []
        batch_tokens = []
        batch_final_pos = []
        max_len_so_far = 0
        for i in range(batch_size):
            example = task[start_i + i]
            assert example['messages'][-1]['role'] == 'assistant'
            batch_examples.append(example)
            tokens, _ = renderer.render_conversation(example['messages'][:-1])  # Cut expected assistant response
            tokens = tokens[:max_seq_len]  # Nanochat hard-truncates to 2048, all ChatCORE categorical fit anyway, see NOTES.md
            tokens.append(assistant_start_token)  # Add <|assistant_start|> to encourage the model
            batch_tokens.append(tokens)
            batch_final_pos.append(len(tokens)-1)  # used to find answer later
            max_len_so_far = max(max_len_so_far, len(tokens))

        # Execute the model
        batch_tokens = [bt + [bos_token] * (max_len_so_far - len(bt)) for bt in batch_tokens]  # pad to max_len_so_far
        x = torch.tensor(batch_tokens, dtype=torch.long, device=device)
        with torch.no_grad():
            logits, _, _ = model(x, return_logits=True)  # B, max_len, V
        logits_pred = logits[range(batch_size), batch_final_pos]  # B,V  select logits at the <|assistant_start|>, where the answer appears

        # Unpack and check answer
        for i in range(len(batch_examples)):
            example = batch_examples[i]
            letter_tokens = [tokenizer.encode_single_token(t) for t in example['eval']['letters']]
            focused_logits = logits_pred[i][letter_tokens]  # [n_letters]   select logits at the position of the letters
            answer_letter_idx = torch.argmax(focused_logits).item()
            model_answer = example['eval']['letters'][answer_letter_idx]
            result = task.evaluate(assistant_response=model_answer, eval_data=example['eval'])
            num_passed += int(result)
            num_total += 1

    # Sync across ranks
    if torch.distributed.is_initialized():
        results_tensor = torch.tensor((num_passed, num_total), dtype=torch.long, device=device)
        torch.distributed.all_reduce(results_tensor, op=torch.distributed.ReduceOp.SUM)
        num_passed, num_total = results_tensor.tolist()
    assert num_total == num_world_total
    
    accuracy = num_passed / num_total  # num_total should never be 0
    return accuracy

In [ ]:
renderer = ConversationRenderer(tokenizer=tokenizer)
task = TaskMMLU("all", "test")
acc = evaluate_sft_categorical(task, model, tokenizer, renderer, device, micro_batch, max_seq_len, max_problems=1000)
print(acc)

0.338

In [69]:
task = TaskArc("ARC-Easy", "test")
acc = evaluate_sft_categorical(task, model, tokenizer, renderer, device, micro_batch, max_seq_len, max_problems=1000)
print(acc)

0.478


In [70]:
task = TaskArc("ARC-Challenge", "test")
acc = evaluate_sft_categorical(task, model, tokenizer, renderer, device, micro_batch, max_seq_len, max_problems=1000)
print(acc)

0.401
